# Data Cleaning with Python

This notebook walks through the same core workflow as `data_cleaning.py`: inspect a raw dataset, check quality issues, remove exact duplicate rows, standardize column names, handle missing values, and export a cleaned CSV.

Run from the repository root. Review whether median/mode imputation is appropriate for your dataset before using it in a real project.

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path("data")
input_file = data_dir / "customer_data.csv"
output_file = data_dir / "customer_data_cleaned.csv"

if not input_file.exists():
    raise FileNotFoundError(f"Dataset not found: {input_file}. Run this notebook from the repository root.")

df = pd.read_csv(input_file)
print(f"Raw shape: {df.shape}")
display(df.head())

## 1. Profile data quality

Check types, missing values, and duplicate rows before transforming the data.

In [ ]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "distinct_values": df.nunique(dropna=True),
})
display(profile)
print(f"Exact duplicate rows: {df.duplicated().sum()}")

## 2. Remove duplicates and standardize column names

These operations are straightforward for a demo dataset. In production, define which fields identify a duplicate business record before dropping rows.

In [ ]:
rows_before = len(df)
df = df.drop_duplicates().copy()
df.columns = (df.columns.astype(str).str.strip().str.lower().str.replace(r"\\s+", "_", regex=True))
print(f"Rows before: {rows_before}; after exact deduplication: {len(df)}")
print("Columns:", df.columns.tolist())

## 3. Handle missing values

For demonstration, fill numeric columns with their median and text/category columns with their mode. All-null columns are left missing because there is no observed value to impute. This is a baseline, not a universal rule.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns
other_cols = df.columns.difference(numeric_cols)

for col in numeric_cols:
    median_value = df[col].median()
    if pd.notna(median_value):
        df[col] = df[col].fillna(median_value)

for col in other_cols:
    modes = df[col].mode(dropna=True)
    if not modes.empty:
        df[col] = df[col].fillna(modes.iloc[0])

display(df.isna().sum().rename("remaining_missing_values").to_frame())

## 4. Validate and export

Check the output before using it downstream. The remaining-missing-values table should be reviewed rather than assumed to be all zeros.

In [ ]:
assert df.columns.is_unique, "Column names must be unique after standardization"
df.to_csv(output_file, index=False)
print(f"Cleaned shape: {df.shape}")
print(f"Saved: {output_file}")
print(f"Remaining duplicate rows: {df.duplicated().sum()}")
print(f"Remaining missing cells: {int(df.isna().sum().sum())}")